# Python Finally & Cleanup

> 📘 **Python Mastery** · Module 07 — Error Handling · Lesson 2/2

Lesson 1 taught your program to survive errors. But surviving is not enough — files must still be closed, connections released, locks freed, whether the run succeeded, exploded, or returned early. `finally` is Python's guarantee that cleanup happens on **every** path. Together with `else`, it completes the full `try`/`except`/`else`/`finally` anatomy used across all professional Python.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- Predict exactly when `finally` runs — including across `return` and re-raised exceptions.
- Lay out the complete `try` / `except` / `else` / `finally` anatomy and assign each block its job.
- Close files safely with `try`/`finally`, and explain why `with` usually replaces it.
- Describe why `return` inside `finally` is a famous trap worth avoiding.
- Recognise resource leaks beyond files (databases, sockets, GPU memory).
- Combine all four clauses in one robust file-reader utility.

## 1. `finally`: The Block That Always Runs

Whatever happens above it — success, a caught exception, even a `return` already decided — the `finally` block executes before leaving the `try` statement. It is how you promise Python: *"this must happen, no matter what."*

**Syntax:**

```python
try:
    risky_work()
except SomeError:
    handle_it()
else:
    celebrate()          # only on success
finally:
    clean_up()           # ALWAYS - success, failure, or return
```

**Example:** the three classic paths, side by side.

In [1]:
print("Case 1: nothing goes wrong")
try:
    tea = "green tea"
except ValueError:
    print("  except -> (skipped)")
else:
    print("  else   -> serving", tea)
finally:
    print("  finally-> table wiped\n")


print("Case 2: something goes wrong")
try:
    raise ValueError("the milk expired")
except ValueError as e:
    print("  except -> handled:", e)
else:
    print("  else   -> (skipped)")
finally:
    print("  finally-> table wiped anyway")

Case 1: nothing goes wrong
  else   -> serving green tea
  finally-> table wiped

Case 2: something goes wrong
  except -> handled: the milk expired
  finally-> table wiped anyway


In [2]:
def take_order():
    try:
        return "coffee"                          # the return is DECIDED here...
    finally:
        print("  finally-> ...but I still ran before it leaves!")   # ...and I run first

drink = take_order()
print("order received:", drink)

  finally-> ...but I still ran before it leaves!
order received: coffee


In [3]:
# finally does NOT shield anyone: the exception pauses,
# finally runs, then the error continues travelling upward.
def level_up():
    try:
        raise RuntimeError("I am still coming!")
    finally:
        print("  finally-> ran on the way through")

try:
    level_up()
except RuntimeError as e:
    print("caller caught:", e)

  finally-> ran on the way through
caller caught: I am still coming!


## 2. Full Anatomy: `try` / `except` / `else` / `finally`

All four clauses form a complete contract. Memorise the table — it doubles as the flowchart:

| Block | Runs when... | Its typical job |
|-------|--------------|-----------------|
| `try` | always attempted | the risky operation itself |
| `except E` | only if that specific error occurred | recover, report, substitute a default |
| `else` | only if `try` raised **nothing** | use the result; success-only work |
| `finally` | **always**, on every exit path | cleanup: close, release, unlock, log |

Order is fixed: `try` → (`except`* | `else`) → `finally`. You may omit `except`/`else`/`finally` individually, but a bare `try` needs at least one companion clause.

**Syntax:**

```python
try:
    open_and_process()       # risk
except (ValueError, OSError) as e:
    report(e)                # known failures
else:
    save_results()           # success only
finally:
    release_resources()      # unconditional
```

**Example:** one function, three inputs, all four stages visible.

In [4]:
def safe_divide(a, b):
    print(f"attempt: safe_divide({a!r}, {b!r})")
    try:
        result = a / b
    except ZeroDivisionError:
        print("  except -> cannot divide by zero")
    except TypeError:
        print("  except -> both operands must be numbers")
    else:
        print(f"  else   -> success, result = {result}")
    finally:
        print("  finally-> attempt finished\n")

safe_divide(10, 4)
safe_divide(10, 0)
safe_divide(10, "4")

attempt: safe_divide(10, 4)
  else   -> success, result = 2.5
  finally-> attempt finished

attempt: safe_divide(10, 0)
  except -> cannot divide by zero
  finally-> attempt finished

attempt: safe_divide(10, '4')
  except -> both operands must be numbers
  finally-> attempt finished



## 3. Real Cleanup: Files Without `with`

Lesson 06 promised that `with` auto-closes files. Here is the machinery underneath, spelled out: open the file, do risky work, and close it in `finally`. Whatever explodes in between, closure is guaranteed — this pattern predates `with` and still powers resources that offer no context manager.

**Syntax:**

```python
f = open(path)               # open BEFORE try (if open itself fails, nothing to clean)
try:
    process(f)
finally:
    f.close()                # guaranteed
```

**Example:** a disaster strikes mid-write; compare with the `with` version below.

In [5]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
log_path = Path("sample_data", "manual_cleanup.log")

log = open(log_path, "w", encoding="utf-8")     # opened outside the try
try:
    log.write("step 1 done\n")
    log.write("step 2 done\n")
    1 / 0                                       # disaster mid-job!
except ZeroDivisionError as e:
    print("handled the disaster:", e)
finally:
    log.close()                                 # ALWAYS runs
    print("file closed?", log.closed)

handled the disaster: division by zero
file closed? True


In [6]:
# The identical guarantee, compressed by 'with'.
try:
    with open("sample_data/auto_cleanup.log", "w", encoding="utf-8") as log:
        log.write("same job, far less ceremony\n")
        1 / 0
except ZeroDivisionError as e:
    print("handled:", e)
# Leaving the with-block - even via exception - already closed the file.

import os
print("both files reached disk:",
      os.path.exists("sample_data/manual_cleanup.log"),
      os.path.exists("sample_data/auto_cleanup.log"))

handled: division by zero
both files reached disk: True True


Closing is what pushes buffered text to disk — so verify the artefact only **after** closure:

**Example:**

In [7]:
from pathlib import Path

# Both logs were closed above -> their buffers reached the disk.
for name in ("manual_cleanup.log", "auto_cleanup.log"):
    print(name, "contains:")
    print(Path("sample_data", name).read_text(encoding="utf-8"))

manual_cleanup.log contains:
step 1 done
step 2 done

auto_cleanup.log contains:
same job, far less ceremony



## 4. Nested Cleanup: Inner `finally` Runs First

`try` blocks can nest — including across function calls — and every layer's `finally` fires on the way out. The innermost cleanup executes **before** any outer handler sees the error, which is exactly how layered software guarantees each layer releases its own resource.

**Syntax:**

```python
try:                        # outer layer
    try:                    # inner layer
        risky()
    finally:
        inner_cleanup()     # ALWAYS runs first
except Exception:
    outer_handler()         # runs AFTER inner_cleanup
```

**Example:**

In [8]:
def inner_job():
    try:
        raise ValueError("data malformed")
    finally:
        print("  inner finally -> temp file deleted")

try:
    inner_job()
except ValueError as e:
    print("outer except   ->", e)
    print("(the inner cleanup already happened before we got here)")

  inner finally -> temp file deleted
outer except   -> data malformed
(the inner cleanup already happened before we got here)


## 5. `finally` With `return`: A Subtle Trap

If `finally` contains a `return`, it **silently discards** whatever `try` was about to return — the function answers with the `finally` value instead. The same applies to `break`/`continue` in loops. No warning, no error: just a different result than every reader expected. Rule: `finally` is for cleanup actions only — never for deciding outcomes.

> 🔍 **Under the Hood:** CPython implements `try`/`finally` with exception-table entries that route *every* exit path through the finally block. A `return` inside `finally` overwrites the in-flight return value on the value stack — which is precisely why linters flag it (`flake8: B012`) and why PEP 765 discourages it outright in recent Pythons.

**Example:** watch a perfectly computed grade get thrown away.

In [9]:
def grade_v1():
    try:
        return "computed grade: A"
    finally:
        return "grade lost!"          # silently wins - NEVER do this

def grade_v2():
    try:
        return "computed grade: A"
    finally:
        print("  (cleanup message only - no return here)")

print("trap   :", grade_v1())
print("correct:", grade_v2())

trap   : grade lost!
  (cleanup message only - no return here)
correct: computed grade: A


## 6. Resource Leaks Beyond Files

Files are just the beginner's example. Every external resource follows the same law: *acquired, must be released*.

- **Database connections** — servers cap them (~100); leaked ones freeze apps until restart.
- **Network sockets / API sessions** — leaked ports and dangling HTTP connections pile up.
- **GPU memory** — unclosed model handles starve the next training job.
- **Locks** — a lock never released deadlocks every thread waiting behind it.

> 🔍 **Under the Hood:** CPython's reference counting usually frees an abandoned object promptly, running its cleanup somewhat by accident. But reference cycles delay it to garbage collection, other interpreters (PyPy) differ, and threads or generators can hold objects alive indefinitely. Professional code never *relies* on timing: it states cleanup explicitly — `finally` or `with`.

**Example:** sketch of the universal shape (pseudo-connection, no server needed).

In [10]:
class FakeDBConnection:
    """A stand-in for a real database driver connection."""
    def query(self, sql):
        print("   querying:", sql)
        return [("Sarah", 92)]
    def close(self):
        print("   connection released")

conn = None
try:
    conn = FakeDBConnection()
    print("rows:", conn.query("SELECT name, score FROM students"))
finally:
    if conn is not None:
        conn.close()          # same discipline as files
print("(Real drivers expose context managers too - prefer 'with'.)")

   querying: SELECT name, score FROM students
rows: [('Sarah', 92)]
   connection released
(Real drivers expose context managers too - prefer 'with'.)


## 7. Choosing the Tool: `finally` vs `with` vs `else`

| Situation | Reach for |
|-----------|-----------|
| Resource supports the context-manager protocol (files, locks, most libraries) | `with` — always first choice |
| Cleanup required around logic with no context manager available | `try` / `finally` |
| Work that must run only when the risky part **succeeded** | `else` |
| Different responses for different failures | `except SpecificError` blocks |
| Hard guarantee: metric logged / connection released on every path | `finally` |

Rule of thumb: **`with` > `finally`** when a manager exists; **`else`** keeps success logic out of the blast radius; **`finally`** is the explicit contract when nothing else fits.

## 8. Putting It All Together: A Robust File Reader

Every idea from Modules 06–07 in one small utility: `pathlib` for paths, specific exceptions for each failure mode, `else` to hand back results only on genuine success, and `finally` guaranteeing closure even when decoding blows up halfway.

**Syntax:**

```python
def robust_read(path):
    f = None
    try:
        f = open(path, encoding="utf-8")
        text = f.read()
    except FileNotFoundError: ...
    except UnicodeDecodeError: ...
    else: return text
    finally:
        if f is not None: f.close()
```

**Example:** three files — healthy, corrupted, absent — and no crash among them.

In [11]:
from pathlib import Path

def robust_read(path):
    """Return a file's content, or a clear diagnosis - never a crash."""
    f = None
    try:
        f = open(path, encoding="utf-8")     # may raise FileNotFoundError
        text = f.read()                       # may raise UnicodeDecodeError
    except FileNotFoundError:
        return "[missing file]"
    except PermissionError:
        return "[no permission]"
    except UnicodeDecodeError as e:
        return f"[not valid UTF-8: {e.reason}]"
    else:
        return text.strip()                   # reached only on full success
    finally:
        if f is not None and not f.closed:
            f.close()
            print(f"   ({Path(path).name} closed in finally)")

In [12]:
from pathlib import Path

good  = Path("sample_data", "diary.txt")
bad   = Path("sample_data", "corrupted.bin")
ghost = Path("sample_data", "ghost.txt")

good.write_text("Dear diary: today I truly understood finally.\n", encoding="utf-8")
bad.write_bytes(b"\xff\xfe high bytes \xff are definitely not UTF-8 text")

for p in (good, bad, ghost):
    print(p.name, "->", robust_read(str(p)))

   (diary.txt closed in finally)
diary.txt -> Dear diary: today I truly understood finally.
   (corrupted.bin closed in finally)
corrupted.bin -> [not valid UTF-8: invalid start byte]
ghost.txt -> [missing file]


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| `return` inside `finally` | Silently overrides the `try` block's return value | Keep `finally` free of `return`/`break`/`continue` |
| Raising new errors inside `finally` | Masks or replaces the original exception | Cleanup should not throw; wrap risky cleanup in its own `try` |
| Assuming `finally` cancels the error | The exception resumes travelling after `finally` runs | Recover in `except`; clean up in `finally` |
| Opening the resource *inside* `try` with manual cleanup | If `open()` itself fails, `finally` references an unborn variable | Open before `try`, or guard with `if f is not None` |
| Hand-writing `try/finally` around a file | Extra code, same guarantee `with` gives | Prefer `with open(...) as f:` |

## 💡 Best Practices & Pro Tips

- Default to `with`; reach for `finally` only when no context manager exists (some raw database drivers, low-level sockets, custom protocols).
- Keep `finally` blocks tiny and boring: close, release, unlock, append one status line. Anything clever belongs elsewhere.
- Use `else` to separate "did it work?" from "use the result" — reviewers can then audit risk and logic independently.
- Remember the pairing rule: **`except` recovers, `else` capitalises, `finally` guarantees.**
- **AI-engineering relevance:** training pipelines live and die by cleanup guarantees — TensorBoard/event-file writers flushed in `finally` after a crashed epoch, GPU memory released between experiments, database connections returned to the pool inside `with` blocks, checkpoints saved "no matter what" before shutdown. Distributed jobs that skip this leak handles until the cluster kills them; the habit starts with today's four-clause anatomy.

## 📌 Summary

| Construct | Guarantees | Example |
|-----------|------------|---------|
| `finally:` | Runs on every exit path — success, failure, `return` | `finally: f.close()` |
| Full anatomy | `try` → `except`* / `else` → `finally`, in that fixed order | see Section 2 table |
| `with` | Context-manager cleanup = automated `finally` | `with open(p) as f:` |
| `finally` + `return` | `finally`'s value silently wins — avoid entirely | `grade_v1()` above |
| Propagation rule | Exceptions pause for `finally`, then continue upward | `level_up()` above |

Key takeaways:

- `finally` is a promise about *your* code's discipline, not a shield against errors.
- `except` heals, `else` enjoys the success, `finally` sweeps the floor — every single time.
- Manual `try`/`finally` is the machinery behind `with`; choose `with` whenever a manager exists.
- Resources beyond files — databases, sockets, GPU memory — deserve the same guaranteed release.

## 🔗 Next Lesson

Your programs now survive their own mistakes. Module 08 teaches them to model the world with objects — starting with classes: [`../../08_OOP/01_Classes/notes.ipynb`](../../08_OOP/01_Classes/notes.ipynb).